In [1]:
from getpass import getuser # Libaray to copy things
from pathlib import Path # Object oriented libary to deal with paths
from dask.utils import format_bytes
from distributed import Client, progress, wait # Libaray to orchestrate distributed resources
#from dask_jobqueue import SLURMCluster # Setting up distributed memories via slurm
import numpy as np # Pythons standard array library
import xarray as xr # Libary to work with labeled n-dimensional data
import dask # Distributed data libary
import dask.distributed
import multiprocessing
from dask.diagnostics import ProgressBar
import glob, os
from matplotlib.lines import Line2D

from matplotlib.patches import Patch
from scipy.ndimage import gaussian_filter1d
from subprocess import run, PIPE
import sys
import os
from sklearn.neighbors import KernelDensity
import gc
from concurrent.futures import ProcessPoolExecutor
import seaborn as sns
import warnings
warnings.filterwarnings(action='ignore')
dask.config.config.get('distributed').get('dashboard').update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})
import pandas as pd
from scipy.interpolate import CloughTocher2DInterpolator, LinearNDInterpolator, NearestNDInterpolator

In [2]:
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter, AutoMinorLocator)
from matplotlib import pyplot as plt # Standard Plotting library
from matplotlib import cm
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter, AutoMinorLocator)
from cartopy import crs as ccrs # Cartography library
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import cartopy.feature as cfeature
import matplotlib.ticker as mticker
import matplotlib.tri as tri

In [3]:
#'''Provides information about the CPU and starts the dask client'''
ncpu = multiprocessing.cpu_count()
processes = False
nworker = 1
threads = ncpu // nworker
print(f"Number of CPUs: {ncpu}, number of threads: {threads}, number of workers: {nworker}, processes: {processes}")
client = Client(processes=processes,
 threads_per_worker=threads,
 n_workers=nworker,
 memory_limit='1000GB'
 )

Number of CPUs: 64, number of threads: 64, number of workers: 1, processes: False


In [4]:
latitude = [30,60]


In [5]:
ifs_cycle4 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/mid_lat_precip/NH/ifs_cycle4_2021-2049_30-60N_precip_total_sum.nc')['tp']
icon_cycle4 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/mid_lat_precip/NH/icon_cycle4_2021-2049_30-60N_precip_total_sum.nc')['pr']
cesm2 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/mid_lat_precip/NH/cesm2_2021-2049_ens_30-60N_precip_total_sum.nc')['PRECT']
#ifs_cycle4 = ifs_cycle4.isel(year=slice(0,30))
#ifs_cycle4 = ifs_cycle4.stack(sample=("year", "time"))
#icon_cycle4 = icon_cycle4.isel(year=slice(0,5))
#icon_cycle4 = icon_cycle4.stack(sample=("year", "time"))



In [6]:
print(ifs_cycle4)

<xarray.DataArray 'tp' (year: 29, value: 574808)> Size: 133MB
[16669432 values with dtype=float64]
Coordinates:
    lon      (value) float64 5MB ...
    lat      (value) float64 5MB ...
Dimensions without coordinates: year, value


In [7]:
print(icon_cycle4)

<xarray.DataArray 'pr' (year: 29, cell: 574808)> Size: 133MB
[16669432 values with dtype=float64]
Coordinates:
    lon      (cell) float64 5MB ...
    lat      (cell) float64 5MB ...
Dimensions without coordinates: year, cell


In [15]:
print(cesm2.time)

<xarray.DataArray 'time' (time: 29)> Size: 232B
array([cftime.DatetimeNoLeap(2021, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2022, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2023, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2024, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2025, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2026, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2027, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2028, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2029, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2030, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2031, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2032, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2033, 1, 1, 0, 0, 0, 0, has_year_ze

In [9]:
land_sea_ifs_zoom9 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_zoom9/30-60N_ifs_hpx512_lsm.nc')['lsm']
land_sea_icon_zoom9 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/NextGEMS/ICON_zoom9/30-60N_ICON_ngc4008_ocean_fraction_surface_zoom9.nc')['ocean_fraction_surface']
land_sea_cesm2 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/CESM2/precip/24H/mask.nc')['MASK']
land_sea_cesm2 = land_sea_cesm2[0,:,:]
indexer = ((land_sea_cesm2.lat>=latitude[0]) & (land_sea_cesm2.lat<=latitude[1]))
land_sea_cesm2 = land_sea_cesm2.where(indexer,drop=True)
land_sea_cesm2 = land_sea_cesm2.assign_coords(lat=cesm2.lat, lon=cesm2.lon)


ifs_cycle4_land = ifs_cycle4.where(land_sea_ifs_zoom9 > 0, np.nan)
icon_cycle4_land = icon_cycle4.where(land_sea_icon_zoom9 < 1, np.nan)
cesm2_land = cesm2.where(land_sea_cesm2 > 0, np.nan)


ifs_cycle4_sea = ifs_cycle4.where(land_sea_ifs_zoom9 == 0, np.nan)
icon_cycle4_sea = icon_cycle4.where(land_sea_icon_zoom9 == 1, np.nan)
cesm2_sea = cesm2.where(land_sea_cesm2 == 0, np.nan)



In [16]:
ifs_hist_land = ifs_cycle4_land.isel(year=slice(0,14))
ifs_fut_land = ifs_cycle4_land.isel(year=slice(14,29))
icon_hist_land = icon_cycle4_land.isel(year=slice(0,14))
icon_fut_land = icon_cycle4_land.isel(year=slice(14,29))
cesm2_hist_land = cesm2_land.isel(time=slice(0,14))
cesm2_fut_land = cesm2_land.isel(time=slice(14,29))

In [21]:
ifs_hist_sea = ifs_cycle4_sea.isel(year=slice(0,14))
ifs_fut_sea = ifs_cycle4_sea.isel(year=slice(14,29))
icon_hist_sea = icon_cycle4_sea.isel(year=slice(0,14))
icon_fut_sea = icon_cycle4_sea.isel(year=slice(14,29))
cesm2_hist_sea = cesm2_sea.isel(time=slice(0,14))
cesm2_fut_sea = cesm2_sea.isel(time=slice(14,29))

In [28]:
print(cesm2_hist_land)

<xarray.DataArray 'PRECT' (member: 10, time: 14, lat: 32, lon: 288)> Size: 10MB
array([[[[ 52.8104555 ,  53.29049701,  49.17107564, ...,  36.22671472,
           44.05928612,  47.02435396],
         [ 54.26781568,  59.09032485,  58.08637145, ...,  35.20158414,
           35.34811761,  41.71281675],
         [ 63.36637034,  66.97046443,  73.13614742, ...,  62.14338492,
           61.47454315,  64.68077191],
         ...,
         [         nan,          nan,          nan, ..., 475.75268014,
                   nan,          nan],
         [         nan,          nan,          nan, ...,          nan,
                   nan,          nan],
         [         nan,          nan,          nan, ...,          nan,
                   nan,          nan]],

        [[ 27.3358713 ,  23.56457364,  19.50079819, ...,  62.47546967,
           58.71550197,  39.34165889],
         [ 41.20614773,  26.4272765 ,  20.29745704, ...,  81.92019053,
           89.52538352,  65.37358969],
         [ 67.53666729, 

In [35]:
ifs_hist_land_mean = ifs_hist_land.mean() / 1000
ifs_fut_land_mean = ifs_fut_land.mean() / 1000
icon_hist_land_mean = icon_hist_land.mean() / 1000
icon_fut_land_mean = icon_fut_land.mean() / 1000
cesm2_hist_land_mean = cesm2_hist_land.mean(dim=['time','lat','lon']) / 1000
cesm2_fut_land_mean = cesm2_fut_land.mean(dim=['time','lat','lon']) / 1000


In [36]:
print(cesm2_hist_land_mean)

<xarray.DataArray 'PRECT' (member: 10)> Size: 80B
array([0.36121269, 0.3621688 , 0.36152802, 0.36383314, 0.36030185,
       0.3594124 , 0.36136434, 0.35697742, 0.36075776, 0.36389859])
Dimensions without coordinates: member


In [38]:
print(cesm2_fut_land_mean)

<xarray.DataArray 'PRECT' (member: 10)> Size: 80B
array([0.36215777, 0.35964611, 0.35899848, 0.36524088, 0.36359886,
       0.36274268, 0.36122897, 0.36452639, 0.35343654, 0.36679704])
Dimensions without coordinates: member


In [37]:
ifs_hist_sea_mean = ifs_hist_sea.mean() / 1000
ifs_fut_sea_mean = ifs_fut_sea.mean() / 1000
icon_hist_sea_mean = icon_hist_sea.mean() / 1000
icon_fut_sea_mean = icon_fut_sea.mean() / 1000
cesm2_hist_sea_mean = cesm2_hist_sea.mean(dim=['time','lat','lon']) / 1000
cesm2_fut_sea_mean = cesm2_fut_sea.mean(dim=['time','lat','lon']) / 1000

In [60]:
ifs_land_diff = ((ifs_fut_land_mean - ifs_hist_land_mean) / ifs_hist_land_mean) * 100
icon_land_diff = ((icon_fut_land_mean - icon_hist_land_mean) / icon_hist_land_mean) * 100
cesm2_land_diff = ((cesm2_fut_land_mean - cesm2_hist_land_mean) / cesm2_hist_land_mean) * 100


In [61]:
ifs_sea_diff = ((ifs_fut_sea_mean - ifs_hist_sea_mean) / ifs_hist_sea_mean) * 100
icon_sea_diff = ((icon_fut_sea_mean - icon_hist_sea_mean) / icon_hist_sea_mean) * 100
cesm2_sea_diff = ((cesm2_fut_sea_mean - cesm2_hist_sea_mean) / cesm2_hist_sea_mean) * 100


In [62]:
print(ifs_land_diff)

<xarray.DataArray 'tp' ()> Size: 8B
array(-3.01335571)


In [64]:

print('land area mean precip diff for ifs cycle4 is ' + str(ifs_land_diff.values) + '%')
print('land area mean precip diff for icon cycle4 is ' + str(icon_land_diff.values) + '%')

print('land area mean precip diff for cesm2 is ' + str(cesm2_land_diff.values) + '%')


land area mean precip diff for ifs cycle4 is -3.0133557131956064%
land area mean precip diff for icon cycle4 is -4.871700143190236%
land area mean precip diff for cesm2 is [ 0.26164    -0.69654976 -0.69968056  0.38692026  0.91507035  0.9265925
 -0.03746013  2.11469093 -2.02940168  0.79650038]%


In [65]:
print('sea area mean precip diff for ifs cycle4 is ' + str(ifs_sea_diff.values) + '%')
print('sea area mean precip diff for icon cycle4 is ' + str(icon_sea_diff.values) + '%')

print('sea area mean precip diff for cesm2 is ' + str(cesm2_sea_diff.values) + '%')

sea area mean precip diff for ifs cycle4 is 2.1934291579030276%
sea area mean precip diff for icon cycle4 is -0.41027065730135875%
sea area mean precip diff for cesm2 is [ 0.63420332  1.6381747   0.88604093 -0.3638454  -0.76907382  0.13236078
 -0.86064267  0.51801373  0.31363555  1.30988009]%


In [66]:
cesm2_land_diff.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_precip/NH/cesm2_ens_precip_diff.nc')


In [68]:
print(cesm2_land_diff.mean())

<xarray.DataArray 'PRECT' ()> Size: 8B
array(0.19383223)
